# Incrementality Testing — Geo Lift (Difference-in-Differences)

## Objective
Estimate the causal impact of a marketing campaign using a geo holdout experiment and difference-in-differences (DiD) approach.

In [14]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf


In [6]:
path = r"C:\Users\user\Incrementality Testing"

In [8]:
df = pd.read_csv(path + "/incrementality_geo_lift_dataset.csv")

df.head()

,week,region,treatment,post_period,spend,conversions,revenue,group,period
0,1,Region_01,1,0,1228.87,175.27,8229.74,Treatment,Pre
1,2,Region_01,1,0,1063.20,181.08,8293.16,Treatment,Pre
2,3,Region_01,1,0,1343.99,185.27,7728.79,Treatment,Pre
3,4,Region_01,1,0,1768.45,190.64,7753.08,Treatment,Pre
4,5,Region_01,1,0,1421.82,211.83,10286.43,Treatment,Pre


In [10]:
df.info()
df.describe()

<class 'pandas.DataFrame'>
RangeIndex: 480 entries, 0 to 479
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   week         480 non-null    int64  
 1   region       480 non-null    str    
 2   treatment    480 non-null    int64  
 3   post_period  480 non-null    int64  
 4   spend        480 non-null    float64
 5   conversions  480 non-null    float64
 6   revenue      480 non-null    float64
 7   group        480 non-null    str    
 8   period       480 non-null    str    
dtypes: float64(3), int64(3), str(3)
memory usage: 33.9 KB


,week,treatment,post_period,spend,conversions,revenue
count,480.000000,480.000000,480.000000,480.000000,480.000000,480.000000
mean,12.500000,0.500000,0.500000,1720.141104,226.512125,9511.220667
std,6.929408,0.500522,0.500522,429.597761,31.511616,1464.532652
min,1.000000,0.000000,0.000000,918.410000,152.610000,5598.600000
25%,6.750000,0.000000,0.000000,1423.072500,202.437500,8416.695000
50%,12.500000,0.500000,0.500000,1574.295000,224.855000,9509.185000
75%,18.250000,1.000000,1.000000,1870.235000,250.097500,10570.795000
max,24.000000,1.000000,1.000000,2947.160000,317.670000,13857.710000


## Average conversions (sanity check)

In [13]:
df.groupby(['group','period'])['conversions'].mean().unstack()

period,Post,Pre
group,,
Control,226.180500,212.882667
Treatment,261.245333,205.740000


Control group change: 226.18 − 212.88 ≈ +13.3 ==> Natural growth (seasonality, trend, etc.)

Treatment group change: 261.25 − 205.74 ≈ +55.5 ==> Growth with campaign

Incremental Lift: Lift = 55.5 − 13.3 ≈ 42.2

In [15]:
# Difference-in-Differences model
model = smf.ols(
    'conversions ~ treatment + post_period + treatment:post_period',
    data=df
).fit()

print(model.summary())

                            OLS Regression Results                            
Dep. Variable:            conversions   R-squared:                       0.460
Model:                            OLS   Adj. R-squared:                  0.457
Method:                 Least Squares   F-statistic:                     135.2
Date:                Thu, 26 Mar 2026   Prob (F-statistic):           2.30e-63
Time:                        10:12:14   Log-Likelihood:                -2188.8
No. Observations:                 480   AIC:                             4386.
Df Residuals:                     476   BIC:                             4402.
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                            coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------
Intercept               212.88

The campaign caused an increase of ~42 conversions per region per week

## The negative treatment coefficient highlights baseline differences between regions, which reinforces the importance of using DiD instead of simple pre-post comparison.

## Incremental conversions (total)

In [16]:
lift = model.params['treatment:post_period']

n_treated_regions = df[df['treatment']==1]['region'].nunique()
n_post_weeks = df[df['post_period']==1]['week'].nunique()

incremental_conversions = lift * n_treated_regions * n_post_weeks

print(incremental_conversions)

5064.8999999999905


## Incremental revenue

In [17]:
avg_revenue_per_conversion = df['revenue'].sum() / df['conversions'].sum()

incremental_revenue = incremental_conversions * avg_revenue_per_conversion

print(incremental_revenue)

212674.62637860957


## Incremental spend

In [18]:
incremental_spend = df[(df['treatment']==1) & (df['post_period']==1)]['spend'].sum()

print(incremental_spend)

287299.89


## Incremental ROI

In [19]:
iroi = incremental_revenue / incremental_spend

print(iroi)

0.7402530727687002


For every 1 dollar spent, the campaign generated  0.74 dollars in incremental revenue. So it is NOT profitable (short-term)
(because iROI < 1)